In [ ]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime, timedelta

# ================================================================
# 1. Utility functions
# ================================================================
def haversine(lon1, lat1, lon2, lat2):
    lon1_rad = np.radians(lon1)
    lat1_rad = np.radians(lat1)
    lon2_rad = np.radians(lon2)
    lat2_rad = np.radians(lat2)
    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad
    a = np.sin(dlat/2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return c * 6371

def calc_distance_deg(lon1, lat1, lon2, lat2):
    return np.sqrt((lon1 - lon2)**2 + (lat1 - lat2)**2)

# ================================================================
# 2. Define time range
# ================================================================
start_time = datetime(2026, 1, 1)
end_time = datetime(2026, 1, 31)

# ================================================================
# 3. Load event data and apply time filtering
# ================================================================
events_raw = pd.read_csv("texnet_events.csv")

# Combine date and time columns
events_raw["time"] = pd.to_datetime(events_raw['Origin Date'] + ' ' + events_raw['Origin Time'])

# Time filtering: keep only 2026-01-01 to 2026-01-31 (inclusive)
events_raw = events_raw[(events_raw["time"] >= start_time) & (events_raw["time"] <= end_time + timedelta(days=1) - timedelta(seconds=1))]
print(f"Events after time filtering: {len(events_raw)}")

# ================================================================
# 4. Load stations and filter active ones
# ================================================================
stations_raw = pd.read_csv("texnet_stations.csv")
stations_raw["Start Date"] = pd.to_datetime(stations_raw["Start Date"])
stations_raw["End Date"] = pd.to_datetime(stations_raw["End Date"], errors='coerce')

stations_active = stations_raw[
    (stations_raw["Start Date"] <= end_time) & 
    ((stations_raw["End Date"].isna()) | (stations_raw["End Date"] >= start_time))
].copy()
stations_active = stations_active.dropna(subset=["Longitude (WGS84)", "Latitude (WGS84)"])

# ================================================================
# 5. Fix the center at SA02
# ================================================================
sa02 = stations_active[stations_active["Station Code"] == "SA02"]
if len(sa02) == 0:
    raise ValueError("Station SA02 not found")
center_lon = sa02.iloc[0]["Longitude (WGS84)"]
center_lat = sa02.iloc[0]["Latitude (WGS84)"]
print(f"Center fixed at SA02: longitude {center_lon:.4f}, latitude {center_lat:.4f}")

# ================================================================
# 6. Set radius
# ================================================================
maxradius_degree = 0.7
print(f"Using radius: {maxradius_degree} deg (~{maxradius_degree * 111:.0f} km)")

# ================================================================
# 7. Spatial filtering
# ================================================================
event_lons = events_raw["Longitude (WGS84)"]
event_lats = events_raw["Latitude (WGS84)"]
distances_deg = calc_distance_deg(center_lon, center_lat, event_lons, event_lats)
mask_events = distances_deg <= maxradius_degree
events_filtered = events_raw[mask_events].copy()

sta_lons = stations_active["Longitude (WGS84)"]
sta_lats = stations_active["Latitude (WGS84)"]
sta_dist_deg = calc_distance_deg(center_lon, center_lat, sta_lons, sta_lats)
mask_stations = sta_dist_deg <= maxradius_degree
stations_filtered = stations_active[mask_stations].copy()

print(f"Events after time filtering: {len(events_raw)}")
print(f"Events inside the circle (Jan 1-30): {len(events_filtered)}")
print(f"Active stations: {len(stations_active)}, stations inside circle: {len(stations_filtered)}")

# ================================================================
# 8. Save config and filtered data
# ================================================================
config = {
    "longitude0": center_lon,
    "latitude0": center_lat,
    "maxradius_degree": maxradius_degree,
    "mindepth": 0,
    "maxdepth": 30,
    "starttime": start_time.isoformat(),
    "endtime": end_time.isoformat(),
    "network": "TX",
    "channel": "HH*,BH*,EH*,HN*",
}

os.makedirs("local/texnet", exist_ok=True)
with open("local/texnet/config.json", "w") as f:
    json.dump(config, f, indent=2)

events_filtered.to_csv("local/texnet/events_filtered.csv", index=False)
stations_filtered.to_csv("local/texnet/stations_filtered.csv", index=False)

print("\n Done!")
print(f"Config file: local/texnet/config.json")
print(f"Event file: local/texnet/events_filtered.csv ({len(events_filtered)} events)")
print(f"Station file: local/texnet/stations_filtered.csv ({len(stations_filtered)} stations)")

In [ ]:
import os
import json
import obspy
from obspy.clients.fdsn.mass_downloader import CircularDomain, Restrictions, MassDownloader

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------
root_path = "local"
region = "texnet"

with open(f"{root_path}/{region}/config.json", "r") as f:
    config = json.load(f)

# ------------------------------------------------------------
# 2. Final list of 27 stations
# ------------------------------------------------------------
selected_stations = [
    'NMP31', 'PB09', 'PB11', 'PB13', 'PB20', 'PB23', 'PB24', 'PB26',
    'PB28', 'PB29', 'PB33', 'PB34', 'PB35', 'PB36', 'PB38', 'PB39',
    'PB40', 'PB43', 'SA01', 'SA02', 'SA04', 'SA06',
    'WB02', 'WB03', 'WB05', 'WB07', 'WB09'
]

# Combine into ObsPy-compatible format (comma-separated)
station_pattern = ",".join(selected_stations)
print(f"Target stations ({len(selected_stations)}): {station_pattern}")

# ------------------------------------------------------------
# 3. Download function
# ------------------------------------------------------------
def download_selected_waveforms():
    # Create storage directory
    waveform_dir = f"{root_path}/{region}/waveforms_27"
    os.makedirs(waveform_dir, exist_ok=True)

    domain = CircularDomain(
        longitude=config["longitude0"],
        latitude=config["latitude0"],
        minradius=0,
        maxradius=config["maxradius_degree"],
    )

    restrictions = Restrictions(
        starttime=obspy.UTCDateTime(config["starttime"]),
        endtime=obspy.UTCDateTime(config["endtime"]),
        chunklength_in_sec=3600 * 24,
        network=None,
        station=station_pattern,
        channel=config.get("channel", "HH*,BH*,EH*,HN*"),
        minimum_interstation_distance_in_m=0,
        minimum_length=0.1,
        reject_channels_with_gaps=False,
        location="*",
    )

    def get_mseed_storage(network, station, location, channel, starttime, endtime):
        folder = starttime.strftime('%Y/%j')
        mseed_name = f"{network}.{station}.{location}.{channel}.mseed"
        filepath = f"{waveform_dir}/{folder}/{mseed_name}"
        if os.path.exists(filepath):
            return True
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        return filepath

    mdl = MassDownloader(providers=["IRIS"])

    print("=" * 70)
    print(f"Downloading waveforms for {len(selected_stations)} stations...")
    print(f"  Time range: {config['starttime']} to {config['endtime']}")
    print(f"  Station list: {selected_stations}")
    print("=" * 70)

    mdl.download(
        domain,
        restrictions,
        mseed_storage=get_mseed_storage,
        stationxml_storage=f"{waveform_dir}/stations",
        download_chunk_size_in_mb=20,
        threads_per_client=3,
        print_report=True,
    )

    print("\n Download complete!")
    print(f"Waveform files saved to: {waveform_dir}")

# ------------------------------------------------------------
# 4. Run
# ------------------------------------------------------------
if __name__ == "__main__":
    download_selected_waveforms()

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ================================================================
# 1. Define the list of 27 stations (downloaded and confirmed)
# ================================================================
station_list = [
    'NMP31', 'PB09', 'PB11', 'PB13', 'PB20', 'PB23', 'PB24', 'PB26',
    'PB28', 'PB29', 'PB33', 'PB34', 'PB35', 'PB36', 'PB38', 'PB39',
    'PB40', 'PB43', 'SA01', 'SA02', 'SA04', 'SA06',
    'WB02', 'WB03', 'WB05', 'WB07', 'WB09'
]

# ================================================================
# 2. Read station data and filter
# ================================================================
stations = pd.read_csv("local/texnet/stations_filtered.csv")
selected_df = stations[stations['Station Code'].isin(station_list)].copy()
selected_df['prefix'] = selected_df['Station Code'].str[:2]

# Sort according to the list order
selected_df['order'] = selected_df['Station Code'].map({code: i for i, code in enumerate(station_list)})
selected_df = selected_df.sort_values('order').drop(columns=['order'])

print(f" Total {len(selected_df)} stations")
print("Station codes:", selected_df['Station Code'].tolist())

# ================================================================
# 3. Save CSV
# ================================================================
selected_df.to_csv("local/texnet/stations_selected_27.csv", index=False)
print("\n Saved: local/texnet/stations_selected_27.csv")

# ================================================================
# 4. Load event data for plotting background
# ================================================================
events = pd.read_csv("local/texnet/events_filtered.csv")
center_lon, center_lat = -104.2649, 31.67163

# ================================================================
# 5. Plot (all 27 stations, color‑coded by prefix)
# ================================================================
fig, ax = plt.subplots(figsize=(10, 8))

# Events (small gray dots)
ax.scatter(events['Longitude (WGS84)'], events['Latitude (WGS84)'],
           c='gray', s=5, alpha=0.5, label='Events')

# SA02 center
ax.scatter(center_lon, center_lat, c='red', marker='*', s=200, label='SA02 Center')

# All candidate stations (light blue triangles, background reference)
candidate_df = stations[stations['Station Code'].isin(station_list)]
ax.scatter(candidate_df['Longitude (WGS84)'], candidate_df['Latitude (WGS84)'],
           marker='^', s=30, color='blue', alpha=0.15, label='All 27 Stations')

# Color by prefix
colors = {'PB': 'red', 'SA': 'orange', 'WB': 'green', 'NM': 'purple'}
for prefix, color in colors.items():
    subset = selected_df[selected_df['prefix'] == prefix]
    if len(subset) > 0:
        ax.scatter(subset['Longitude (WGS84)'], subset['Latitude (WGS84)'],
                   marker='^', s=120, color=color, label=f'{prefix} ({len(subset)})')

# Add station labels
for _, row in selected_df.iterrows():
    ax.annotate(row['Station Code'], 
                (row['Longitude (WGS84)'], row['Latitude (WGS84)']),
                fontsize=8, xytext=(3, 3), textcoords='offset points')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend()
ax.set_title('Downloaded 27 Stations (1–30 Jan 2026)')
ax.grid(True, linestyle='--', alpha=0.3)

# Expand x‑axis range to make NMP31 (longitude ≈ -103.596°) visible
ax.set_xlim(center_lon - 0.7, center_lon + 0.7)
ax.set_ylim(center_lat - 0.5, center_lat + 0.5)

# ================================================================
# 6. Save figure
# ================================================================
output_dir = "local/texnet/figures"
os.makedirs(output_dir, exist_ok=True)
plt.savefig(os.path.join(output_dir, "all_27_stations.png"), dpi=300, bbox_inches='tight')
print(f" Figure saved: {output_dir}/all_27_stations.png")
plt.show()

# ================================================================
# 7. Print summary statistics
# ================================================================
print("\n" + "="*60)
print("Station statistics (27 stations)")
print("="*60)
for prefix in ['PB', 'SA', 'WB']:
    count = len(selected_df[selected_df['prefix'] == prefix])
    print(f"  {prefix}: {count}")
other = len(selected_df[~selected_df['prefix'].isin(['PB','SA','WB'])])
print(f"  Other (NMP): {other}")
print("="*60)